# 03 — Translation & Cleaning (Olist reviews)

**Day 3 — May 21, 2026**

Goal: translate a stratified sample of Portuguese Olist reviews to English so
that all NLP work runs on the **same customers** as the transactional spine
(`olist_master.parquet`). Output stays keyed on `order_id` for a true row-level
join back to orders/customers/payments.

Decision recap: dropped Women's Clothing from the pipeline to preserve
customer-level linkage. Olist is now the single source for both transactions
and text.

In [1]:
import time
from pathlib import Path

import pandas as pd

PROCESSED = Path("data/processed")
REVIEWS_IN = PROCESSED / "olist_reviews_with_lang.parquet"
MASTER_IN = PROCESSED / "olist_master.parquet"
CHECKPOINT = PROCESSED / "olist_translation_checkpoint.parquet"
FINAL_OUT = PROCESSED / "olist_reviews_translated.parquet"

In [2]:
from pathlib import Path

# auto-detect root whether we're in project root or notebooks/
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
PROCESSED = ROOT / "data" / "processed"

REVIEWS_IN = PROCESSED / "olist_reviews_with_lang.parquet"
MASTER_IN  = PROCESSED / "olist_master.parquet"
CHECKPOINT = PROCESSED / "olist_translation_checkpoint.parquet"
FINAL_OUT  = PROCESSED / "olist_reviews_translated.parquet"

print("PROCESSED:", PROCESSED)
print("Exists:", PROCESSED.exists())
print(list(PROCESSED.iterdir()))

PROCESSED: C:\Users\akskumari\Desktop\cx-analytics-project\data\processed
Exists: True
[WindowsPath('C:/Users/akskumari/Desktop/cx-analytics-project/data/processed/.gitkeep'), WindowsPath('C:/Users/akskumari/Desktop/cx-analytics-project/data/processed/olist_master.parquet'), WindowsPath('C:/Users/akskumari/Desktop/cx-analytics-project/data/processed/olist_reviews_with_lang.parquet'), WindowsPath('C:/Users/akskumari/Desktop/cx-analytics-project/data/processed/olist_translation_checkpoint.parquet'), WindowsPath('C:/Users/akskumari/Desktop/cx-analytics-project/data/processed/womens_clean.parquet')]


## Step 0 — VERIFY COLUMN NAMES FIRST (do not skip)

The code below assumes these column names. Print the real ones and fix the
constants if they differ, otherwise everything downstream breaks silently.

In [3]:
reviews = pd.read_parquet(REVIEWS_IN)
print("Columns in olist_reviews_with_lang.parquet:")
print(reviews.columns.tolist())
print("\nShape:", reviews.shape)
reviews.head(3)

Columns in olist_reviews_with_lang.parquet:
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp', 'response_time_days', 'language']

Shape: (40577, 9)


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,response_time_days,language
0,b15195a1b61f6dd139324238334dd67b,788d6da30a84d084503d9e25fa8cee57,5,Pendente lindo,Realizei uma compra desse pendente e quando ch...,2018-08-31,2018-09-03 14:47:28,3,pt
1,7b3f3d737b4cce3a7c03506bae15d187,ddba311fbcd3b55e2964aadd8ca8d231,5,Produto excelente!,Um excelente produto e entrega rápida! Apenas ...,2018-08-31,2018-09-01 14:33:29,1,pt
2,9e25d6e3025e9b9a0fc7f03588d33e2b,869997fbe01f39d184956b5c6bccfdbe,1,Razoavel,Peço um produto por um código e vem outro tota...,2018-08-31,2018-09-03 09:33:00,3,pt


In [4]:
# --- EDIT THESE to match what printed above ---
COL_ID = "order_id"
COL_TEXT = "review_comment_message"
COL_LANG = "language"          # column from your langdetect step
COL_SCORE = "review_score"

# sanity: confirm they exist
for c in (COL_ID, COL_TEXT, COL_LANG, COL_SCORE):
    assert c in reviews.columns, f"MISSING COLUMN: {c} — fix the constant above"
print("All expected columns present.")

All expected columns present.


## Step 1 — Inspect language + score distribution
Confirms how much Portuguese text we actually have and where the negative
reviews sit (those are the signal we care most about).

In [5]:
has_text = reviews[COL_TEXT].notna() & (reviews[COL_TEXT].astype(str).str.len() > 0)
print("Reviews with text:", has_text.sum())
print("\nLanguage distribution (text only):")
print(reviews.loc[has_text, COL_LANG].value_counts().head(10))
print("\nScore distribution (Portuguese, text only):")
pt_mask = has_text & (reviews[COL_LANG] == "pt")
print(reviews.loc[pt_mask, COL_SCORE].value_counts().sort_index())

Reviews with text: 40577

Language distribution (text only):
language
pt         34258
unknown     3179
it          1207
es           725
ro           193
sk           178
en           159
de           130
ca           122
sl            80
Name: count, dtype: int64

Score distribution (Portuguese, text only):
review_score
1     8317
2     2005
3     3101
4     4732
5    16103
Name: count, dtype: int64


## Step 2 — Build stratified sample (~12K, oversample negatives)
We translate a sample, not all ~40K: free translation rate-limits hard and we
only need enough for classifier training + topic modeling. We deliberately
take ALL 1–2 star reviews (the complaint signal) and fill the rest from 3–5.

In [6]:
SAMPLE_SIZE = 12_000

pt = reviews.loc[pt_mask].copy()
neg = pt[pt[COL_SCORE] <= 2]
rest = pt[pt[COL_SCORE] > 2]
take_rest = max(0, SAMPLE_SIZE - len(neg))
rest_s = rest.sample(min(take_rest, len(rest)), random_state=42)

sample = pd.concat([neg, rest_s]).drop_duplicates(COL_ID).head(SAMPLE_SIZE)
print(f"Sample size: {len(sample)}")
print(f"  negative (<=2 star): {(sample[COL_SCORE] <= 2).sum()}")
print(f"  rest (>2 star):      {(sample[COL_SCORE] > 2).sum()}")

Sample size: 12000
  negative (<=2 star): 10322
  rest (>2 star):      1678


## Step 3 — Translate PT → EN (resumable)

**Recommended:** run this as the standalone `src/translate_reviews.py` in a
terminal for the long run (it checkpoints every 200 rows, survives crashes).
Then skip to Step 4 and just load the result.

If you'd rather run it here, the cell below does the same thing inline. It's
slower and more fragile in a notebook, but fine for a smaller sample.

In [7]:
from deep_translator import GoogleTranslator

BATCH_SLEEP = 0.4   # politeness delay between calls
SAVE_EVERY = 200    # checkpoint frequency


def load_done_ids():
    if CHECKPOINT.exists():
        return set(pd.read_parquet(CHECKPOINT)[COL_ID].tolist())
    return set()


def flush(rows):
    new = pd.DataFrame(rows)
    if CHECKPOINT.exists():
        old = pd.read_parquet(CHECKPOINT)
        new = pd.concat([old, new]).drop_duplicates(COL_ID)
    new.to_parquet(CHECKPOINT, index=False)


done = load_done_ids()
todo = sample[~sample[COL_ID].isin(done)]
print(f"Already done: {len(done)} | Remaining: {len(todo)}")

translator = GoogleTranslator(source="pt", target="en")
rows, since_save = [], 0

for i, (_, r) in enumerate(todo.iterrows()):
    src = str(r[COL_TEXT])[:4900]  # API char cap safety
    try:
        en = translator.translate(src)
    except Exception as e:
        print(f"  ! fail {r[COL_ID]}: {e} — sleep 5s, retry")
        time.sleep(5)
        try:
            en = translator.translate(src)
        except Exception:
            en = None
    rows.append({COL_ID: r[COL_ID], "review_pt": r[COL_TEXT],
                 "review_en": en, COL_SCORE: r[COL_SCORE]})
    since_save += 1
    time.sleep(BATCH_SLEEP)
    if since_save >= SAVE_EVERY:
        flush(rows)
        rows, since_save = [], 0
        print(f"  checkpoint @ {i + 1}/{len(todo)}")

if rows:
    flush(rows)
print("Translation pass complete.")

Already done: 12000 | Remaining: 0
Translation pass complete.


## Step 4 — Load translated result + spot-check quality
Eyeball 30–40 translations. Note any garbage (very short slang, typos that
don't translate). Log them but don't over-clean — a few rough ones are fine.

In [8]:
translated = pd.read_parquet(CHECKPOINT)
print("Translated rows:", len(translated))
print("Null/failed translations:", translated["review_en"].isna().sum())

# random spot-check
translated.dropna(subset=["review_en"]).sample(min(30, len(translated)),
                                                random_state=1)[
    ["review_pt", "review_en", COL_SCORE]
]

Translated rows: 12000
Null/failed translations: 3


,review_pt,review_en,review_score
11867,Produto exatamente como descrito no anúncio de...,Product exactly as described in the sales adve...,5
7648,"Boa tarde, não recebi as pétalas, somente o ad...","Good afternoon, I didn't receive the petals, o...",1
8513,"esse produto veio todo torto, o encache das pe...","This product came all crooked, the fitting of ...",1
8286,meu produto não foi entregue ainda ..,my product has not been delivered yet..,1
10588,Boa compra,Good buy,5
6439,Compra efetuada em 28/11 e a nota fiscal ainda...,Purchase made on 11/28 and the invoice has not...,1
6596,Atraso na entrega sem dar satisfação ao client...,Delay in delivery without satisfying the custo...,1
7737,Produto entregue pela metade so recebi a mochi...,"Product delivered in half, I only received the...",1
5001,O produto não veio com o m\nual de montagem.,The product did not come with an assembly manual.,2
2070,Não recebi os buchinhos só veio a nota fiscal ...,"I didn't receive the buchinhos, only the invoi...",1


## Step 5 — Apply Day 2 cleaning functions

Reuse the two cleaners you built on Women's Clothing (this is where that work
pays off). Paste your real `clean_text` and `preprocess_text` here, or import
them from `src/text_preprocessing.py`. Placeholders below — REPLACE with yours.

In [9]:
import sys
sys.path.append(str(ROOT))  # makes src/ importable from notebooks/
from src.text_preprocessing import clean_text, preprocess_text

In [10]:
df = translated.dropna(subset=["review_en"]).copy()
df["review_clean"] = df["review_en"].apply(clean_text)
df["review_preprocessed"] = df["review_en"].apply(preprocess_text)
print("Cleaned rows:", len(df))
df[["review_en", "review_clean", "review_preprocessed"]].head(5)

Cleaned rows: 11997


,review_en,review_clean,review_preprocessed
0,I order a product using a code and a completel...,I order a product using a code and a completel...,order product using code completely different ...
1,"I don't recommend it, in addition to the delay...","I don't recommend it, in addition to the delay...",recommend addition delay delivery one chair ar...
2,I didn't receive,I didn't receive,receive
3,"They only delivered one item of the purchase, ...","They only delivered one item of the purchase, ...",delivered one item purchase fish grill
4,"I bought three rolls of wallpaper, I only rece...","I bought three rolls of wallpaper, I only rece...",bought three roll wallpaper received get touch


## Step 6 — Before/after documentation samples
Show the full pipeline on a few rows for the portfolio narrative:
raw PT → translated EN → light clean → aggressive preprocess.

In [11]:
for _, r in df.sample(3, random_state=7).iterrows():
    print("PT  :", str(r["review_pt"])[:160])
    print("EN  :", str(r["review_en"])[:160])
    print("CLEAN:", str(r["review_clean"])[:160])
    print("PREP:", str(r["review_preprocessed"])[:160])
    print("-" * 70)

PT  : O produto veio como descrito na página e chegou bem antes do prazo determinado. Recomendo!
EN  : The product came as described on the page and arrived well before the deadline. I recommend!
CLEAN: The product came as described on the page and arrived well before the deadline. I recommend!
PREP: product came described page arrived well deadline recommend
----------------------------------------------------------------------
PT  : recebemos somente 02 cadeiras
EN  : We only received 2 chairs
CLEAN: We only received 2 chairs
PREP: received chair
----------------------------------------------------------------------
PT  : Produto consta como entregue aos correios do RJ desde o dia 20/03 e ainda não saiu para entrega.
EN  : Product appears to have been delivered to the RJ post office since 03/20 and has not yet been released for delivery.
CLEAN: Product appears to have been delivered to the RJ post office since 03/20 and has not yet been released for delivery.
PREP: product appears de

## Step 7 — Save, keyed on order_id for linkage
This file joins straight back to olist_master.parquet on order_id.

In [12]:
FINAL_OUT.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(FINAL_OUT, index=False)
size_mb = FINAL_OUT.stat().st_size / 1024 / 1024
print(f"Saved {FINAL_OUT.name} ({size_mb:.1f} MB), {len(df)} rows")

# verify the join works
master = pd.read_parquet(MASTER_IN)
joined = df.merge(master[[COL_ID]], on=COL_ID, how="inner")
print(f"Rows that join to master on {COL_ID}: {len(joined)} / {len(df)}")

Saved olist_reviews_translated.parquet (2.8 MB), 11997 rows
Rows that join to master on order_id: 11997 / 11997


In [16]:
from pathlib import Path
for f in sorted(Path(r"C:\Users\akskumari\Desktop\cx-analytics-project\data\processed").iterdir()):
    print(f.name, round(f.stat().st_size/1024/1024, 1), "MB")

.gitkeep 0.0 MB
olist_master.parquet 20.7 MB
olist_reviews_translated.parquet 2.8 MB
olist_reviews_with_lang.parquet 4.8 MB
womens_clean.parquet 10.5 MB
